# Advanced Machine Learning - Final Project
## Notebook 2: Data Integration and Multi-Class Labeling

### 1. Parsing Laundering Topologies
The `HI-Medium_Patterns.txt` file contains the actual illicit transactions, grouped by the specific money laundering topology (Fan-In, Cycle, Stack...). Before we can merge this data with our main transaction dataset, we must parse the text file to extract both the transaction details and the overarching laundering pattern they belong to.

In [1]:
import pandas as pd
from pathlib import Path

PATTERNS_PATH = Path("data/HI-Medium_Patterns.txt")

def get_base_pattern(header: str) -> str:
    """
    Extracts the core laundering topology from the text header.
    The header usually follows 'BEGIN LAUNDERING ATTEMPT -'
    """
    h = header.upper().strip()

    # Order matters: we must check for composite names first to avoid partial matches
    if "GATHER-SCATTER" in h and "SCATTER-GATHER" not in h:
        return "GATHER-SCATTER"
    if "SCATTER-GATHER" in h and "GATHER-SCATTER" not in h:
        return "SCATTER-GATHER"

    if "STACK" in h:
        return "STACK"
    if "FAN-IN" in h or "FAN IN" in h:
        return "FAN-IN"
    if "FAN-OUT" in h or "FAN OUT" in h:
        return "FAN-OUT"
    if "BIPARTITE" in h:
        return "BIPARTITE"
    if "CYCLE" in h:
        return "CYCLE"
    if "RANDOM" in h:
        return "RANDOM"

    return "UNKNOWN"


def parse_patterns_with_base(patterns_path: Path) -> pd.DataFrame:
    """
    Reads the raw patterns text file and constructs a structured DataFrame.
    It injects a new column 'pattern_base' into every transaction indicating
    the type of laundering ring it belongs to.
    """
    rows = []
    current_pattern_base = None

    with open(patterns_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Detect the start of a new laundering ring
            if line.startswith("BEGIN LAUNDERING ATTEMPT -"):
                header = line.split("BEGIN LAUNDERING ATTEMPT -", 1)[1].strip()
                current_pattern_base = get_base_pattern(header)
                continue

            # Detect the end of the current laundering ring
            if line.startswith("END LAUNDERING ATTEMPT -"):
                current_pattern_base = None
                continue

            # If we are inside a pattern block and the line contains commas, it is a transaction
            if current_pattern_base is not None and "," in line:
                parts = [p.strip() for p in line.split(",")]
                
                # We expect exactly 11 columns in the raw data string
                if len(parts) < 11:
                    continue  # Ignore malformed rows

                rows.append({
                    "timestamp": parts[0],
                    "source_id": parts[1],
                    "source_account": parts[2],
                    "destination_id": parts[3],
                    "destination_account": parts[4],
                    "amount": float(parts[5]),
                    "currency_out": parts[6],
                    "amount_in": float(parts[7]),
                    "currency_in": parts[8],
                    "method": parts[9],
                    "flag": int(parts[10]),
                    "pattern_base": current_pattern_base,
                })

    return pd.DataFrame(rows)

print("Parsing pattern topologies...")
patterns_df = parse_patterns_with_base(PATTERNS_PATH)

print("\n--- Parsed Patterns Preview ---")
display(patterns_df.head())

print("\n--- Detected Laundering Classes ---")
print(patterns_df["pattern_base"].value_counts())

Parsing pattern topologies...

--- Parsed Patterns Preview ---


,timestamp,source_id,source_account,destination_id,destination_account,amount,currency_out,amount_in,currency_in,method,flag,pattern_base
0,2022/09/01 05:14,00952,8139F54E0,0111632,8062C56E0,5331.44,US Dollar,5331.44,US Dollar,ACH,1,STACK
1,2022/09/03 13:09,0111632,8062C56E0,008456,81363F620,5602.59,US Dollar,5602.59,US Dollar,ACH,1,STACK
2,2022/09/01 07:40,0118693,823D5EB90,013729,801CF2E60,1400.54,US Dollar,1400.54,US Dollar,ACH,1,STACK
3,2022/09/01 14:19,013729,801CF2E60,0123621,81A7090F0,1467.94,US Dollar,1467.94,US Dollar,ACH,1,STACK
4,2022/09/02 12:40,0024750,81363F410,0213834,808757B00,16898.29,US Dollar,16898.29,US Dollar,ACH,1,STACK



--- Detected Laundering Classes ---
pattern_base
GATHER-SCATTER    4289
SCATTER-GATHER    3988
STACK             3986
FAN-IN            2315
CYCLE             2235
BIPARTITE         2135
FAN-OUT           2128
RANDOM            1667
Name: count, dtype: int64


### 2. Schema Alignment and Data Integration
To train our machine learning models, we need a single, unified dataset. 

We will perform a **Left Join** between the massive main transaction dataset (which contains mostly normal behavior) and our parsed `patterns_df`. We match them using a composite key consisting of the sender/receiver IDs, accounts, amounts and payment formats. 

Before merging, we must ensure strict schema and data type alignment to prevent pandas from generating duplicate rows or failing to match equivalent transactions.

In [2]:
# Load the main transaction dataset
DATA_PATH = Path("data/HI-Medium_Trans.csv")
print("Loading main transaction dataset...")
df = pd.read_csv(DATA_PATH)

print(f"Main dataset shape: {df.shape}")

# Rename the parsed pattern columns to perfectly match the main dataset schema
patterns_merge = patterns_df.rename(columns={
    "source_id": "From Bank",
    "source_account": "Account",
    "destination_id": "To Bank",
    "destination_account": "Account.1",
    "amount": "Amount Received",
    "method": "Payment Format",
})

# Isolate only the keys needed for the merge, plus our target label
patterns_merge = patterns_merge[
    ["From Bank", "Account", "To Bank", "Account.1", "Amount Received", "Payment Format", "pattern_base"]
].copy()

# Enforce strict data type alignment
# The main CSV uses integers for Bank IDs, so we cast our parsed strings to match
patterns_merge["From Bank"] = patterns_merge["From Bank"].astype(int)
patterns_merge["To Bank"]   = patterns_merge["To Bank"].astype(int)

# Define the composite key for the Left Join
merge_keys = ["From Bank", "Account", "To Bank", "Account.1", "Amount Received", "Payment Format"]

print("\nExecuting Left Join...")
# Execute the merge: Normal transactions will get a NaN in 'pattern_base', 
# while illicit ones will receive their specific topology label.
df_merged = df.merge(
    patterns_merge,
    on=merge_keys,
    how="left"
)

print(f"Merged dataset shape: {df_merged.shape}")
print("\n--- Pattern Distribution after Merge (Including Normal Transactions) ---")
print(df_merged["pattern_base"].value_counts(dropna=False))

Loading main transaction dataset...
Main dataset shape: (31898238, 11)

Executing Left Join...
Merged dataset shape: (31898238, 12)

--- Pattern Distribution after Merge (Including Normal Transactions) ---
pattern_base
NaN               31875495
GATHER-SCATTER        4289
SCATTER-GATHER        3988
STACK                 3986
FAN-IN                2315
CYCLE                 2235
BIPARTITE             2135
FAN-OUT               2128
RANDOM                1667
Name: count, dtype: int64


### 3. Target Encoding and Export
Machine learning algorithms require numerical target variables. We map our string-based laundering typologies to a discrete integer space `[1, 8]`. All normal transactions (currently represented by `NaN` resulting from the Left Join) are filled with `0`.

Finally, we export the structured, labeled dataset to be used in our modeling pipeline.

In [3]:
# Define the integer mapping dictionary for our multi-class target
pattern_to_id = {
    "STACK": 1,
    "CYCLE": 2,
    "FAN-IN": 3,
    "FAN-OUT": 4,
    "GATHER-SCATTER": 5,
    "SCATTER-GATHER": 6,
    "BIPARTITE": 7,
    "RANDOM": 8,
}

print("Encoding target variable...")
# Map the strings to integers
df_merged["target_multi"] = df_merged["pattern_base"].map(pattern_to_id)

# Fill unmapped transactions (the normal majority class) with 0 and cast to integer
df_merged["target_multi"] = df_merged["target_multi"].fillna(0).astype(int)

print("\n--- Final Multi-Class Target Distribution (0 = Normal) ---")
print(df_merged["target_multi"].value_counts().sort_index())

# Save the final preprocessed dataset
output_file = "data/ibm_aml_multiclass_clases.csv"
print(f"\nExporting dataset to: {output_file}")
df_merged.to_csv(output_file, index=False)
print("Export complete")

Encoding target variable...

--- Final Multi-Class Target Distribution (0 = Normal) ---
target_multi
0    31875495
1        3986
2        2235
3        2315
4        2128
5        4289
6        3988
7        2135
8        1667
Name: count, dtype: int64

Exporting dataset to: data/ibm_aml_multiclass_clases.csv
Export complete
